# Additional Tasks for RS FMs

In [1]:
import torch
import math
from redundant_features import SCALES

def select_four_layers(features: list[torch.Tensor], img_size: int, patch_size: int) -> torch.Tensor:
    """
    Args:
        features:   List of tensors, each of shape (B, N, D) or (B, D, H_p, W_p)
        img_size:   Input image size (assumes square)
        patch_size: Patch size used by the backbone (assumes square)
    Returns:
        Tensor of shape (B, 4, D, H_p, W_p)
        where H_p = W_p = img_size // patch_size
    """
    n_patches_per_side = img_size // patch_size
    n_patches = n_patches_per_side ** 2
    n = len(features)
    indices = [0, 0, 0, 0] if n == 1 else [round(i * (n - 1) / 3) for i in range(4)]

    first = features[0]
    if first.dim() == 4:
        # Input is (B, D, H_p, W_p)
        stacked = torch.stack([features[i] for i in indices], dim=1)  # (B, 4, D, H_p, W_p)
    else:
        # Input is (B, N, D)
        n_tokens = first.shape[1]
        assert n_tokens in (n_patches, n_patches + 1), (
            f"Unexpected token count {n_tokens}, expected {n_patches} or {n_patches + 1}"
        )
        has_cls = n_tokens == n_patches + 1
        stacked = torch.stack([features[i] for i in indices], dim=1)  # (B, 4, N, D)
        if has_cls:
            stacked = stacked[:, :, 1:, :]  # (B, 4, P*P, D)
        B, four, _, D = stacked.shape
        stacked = stacked.reshape(B, four, n_patches_per_side, n_patches_per_side, D)
        stacked = stacked.permute(0, 1, 4, 2, 3)  # (B, 4, D, H_p, W_p)

    return stacked.contiguous()

In [2]:
BASE_PATH = "./_features/SlimAttn+SlimFFN"
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
def get_multiscale_features_and_labels(model, dm, mode='train'):
    assert mode in ['train', 'val', 'test'], f"mode has to be in ['train', 'val', 'test'] but was {mode}"
    pl.seed_everything(42, verbose=False)
    model.to(device)
    model.eval()
    inputs = []
    labels = []
    if isinstance(dm.val_dataloader(), torch.utils.data.DataLoader):
        val_dl = dm.val_dataloader()
    else:
        val_dl = dm.val_dataloader()[0]
    dl = dm.train_dataloader() if mode == 'train' else val_dl if mode == 'val' else dm.test_dataloader()
    print_proc_unknown = True
    #for i, batch in tqdm(enumerate(dl), desc="Extracting features", total=len(dl), leave=False):
    for i, batch in enumerate(dl):
        x = batch['input'].to(device)
        y = batch['label']

        with torch.no_grad():
            features = model(x)
            features = select_four_layers(features, img_size=224, patch_size=ps)

        features = torch.nn.functional.normalize(features, dim=-1)
        features = features.cpu()
        inputs.append(features)
        labels.append(y)

    all_features = torch.cat(inputs, dim=0).cpu().numpy()
    all_labels = torch.cat(labels, dim=0).cpu().numpy()

    return all_features, all_labels

def extract_multiscale_features(model, dm, model_name, dataset_name):
    for scale in tqdm(SCALES, desc=f"Iterating scales"):
        #print(f"{model_name} : {scale}")
        for mode in ['train', 'val', 'test']:
            # sometimes the backbone, sometimes the main model has this attribute
            try:
                model.backbone.set_to_scale(scale)
            except AttributeError:
                try:
                    model.set_to_scale(scale)    
                except AttributeError:                    
                    model.model.set_to_scale(scale)
            features, labels = get_multiscale_features_and_labels(model, dm, mode=mode)
            Path(f"{BASE_PATH}/{model_name}/scale_{scale}/").mkdir(exist_ok=True, parents=True)
            np.save(f"{BASE_PATH}/{model_name}/scale_{scale}/features_{dataset_name}_{mode}.npy", features)
            np.save(f"{BASE_PATH}/{model_name}/scale_{scale}/labels_{dataset_name}_{mode}.npy", labels)

## Segmentation

In [4]:
import geobench
from tqdm import tqdm
import lightning.pytorch as pl
from terratorch import BACKBONE_REGISTRY
from terratorch_baselines import evaluated_models
from terratorch_baselines import bandorders as terratorch_baseline_bandorders
from terratorch_slimable import make_terratorch_prithvi_dofa_slimable, make_terratorch_terramind_slimable
from geobench_evaluate import get_segmentation_transform
from operator import itemgetter
from pathlib import Path
import numpy as np
from redundant_features import SCALES

extract_segmentation = False
datasets = ['m-cashew-plant', 'm-SA-crop-type']

if extract_segmentation:
    for dataset in datasets:
        geobench_task = [t for t in geobench.task_iterator(benchmark_name="segmentation_v1.0") if t.dataset_name == dataset][0]
        geobench_dataset = geobench_task.get_dataset(split="train")
        
        for name_or_path in ['copernicusFM','panopticon']:
            ps = 14 if name_or_path in ['prithvi2_600', 'panopticon'] else 16
            batch_size = 64
            
            bandnames = itemgetter(*terratorch_baseline_bandorders[name_or_path]["segmentation_v1.0"][dataset])(
                geobench_dataset.task_specs.bands_info)
            bandnames_param = [bn.name for bn in bandnames]
            if 'Cloud Probability' in bandnames_param:
                # cloud prob does not work, use 12-SWIR as placeholder band -> will be removed in KNN & linear as dublicate
                bandnames_param[bandnames_param.index('Cloud Probability')] = '12 - SWIR'
            
            dm = geobench_task.get_pytorch_data_module(
                band_names=bandnames_param,
                train_transform=get_segmentation_transform(geobench_task, bandnames_param, train=True, input_size=224),
                eval_transform=get_segmentation_transform(geobench_task, bandnames_param, train=False, input_size=224),
                batch_size=batch_size,
                num_workers=16,
            )
            
            version = name_or_path
            if name_or_path == 'panopticon':
                import panopticon_helpers
                from torchgeo.models import panopticon_vitb14, Panopticon_Weights

                keep_idx, chn_ids = [], []
                for i, b in enumerate(bandnames):
                    cid = panopticon_helpers.panopticon_chn_id(b, include_sar=False)
                    if cid is None:
                        continue
                    keep_idx.append(i)
                    chn_ids.append(cid)
                assert keep_idx, f"no usable Panopticon channels for {dataset}"
                
                pano = panopticon_vitb14(weights=Panopticon_Weights.VIT_BASE14, img_size=224)
                backbone = panopticon_helpers.PanopticonBackbone(pano.model, chn_ids,
                                                                 feature_mode="blocks")
                backbone = panopticon_helpers.make_panopticon_slimable(backbone)
            elif name_or_path == 'copernicusFM':
                import copernicusFM_helpers
                from copernicusFM import vit_base_patch16
                COPERNICUSFM_CHECKPOINTS = [
                    ("/home/leonard/data/model-weights/CopernicusFM_ViT_base_varlang_e100.pth", "copernicusFM"),
                    ("/scratch/htc/lhackel/model-weights/CopernicusFM_ViT_base_varlang_e100.pth", "copernicusFM"),
                ]
    
                checkpoint_path, version = next(
                    ((Path(c), n) for c, n in COPERNICUSFM_CHECKPOINTS if Path(c).exists()),
                    (None, None)
                )
    
                if checkpoint_path is None:
                    raise FileNotFoundError(f"No CopernicusFM checkpoint found in: {COPERNICUSFM_CHECKPOINTS}")
                checkpoint_path = checkpoint_path.resolve(strict=True)
    
                keep_idx, wvs, bws = [], [], []
                for i, b in enumerate(bandnames):
                    params = copernicusFM_helpers.copernicusFM_band_params(b, include_sar=False)
                    if params is None:
                        continue
                    keep_idx.append(i)
                    wv, bw = params
                    wvs.append(wv)
                    bws.append(bw)
                assert keep_idx, f"no usable CopernicusFM channels for {dataset}"
    
                backbone = vit_base_patch16(num_classes=10, global_pool=False)
                checkpoint = torch.load(checkpoint_path, map_location="cpu")
                state_dict = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint
                msg = backbone.load_state_dict(state_dict, strict=False)
                #print(msg)
    
                backbone = make_terratorch_prithvi_dofa_slimable(backbone)  # CopernicusFM is similar enough to DOFA
                backbone = copernicusFM_helpers.CopernicusFMBackbone(backbone, wvs, bws, kernel_size=16, input_mode="spectral")
                backbone.get_all_features(True)
            else:
                backbone = BACKBONE_REGISTRY.build(**evaluated_models[name_or_path]["kwargs"])
                if "prithvi" in name_or_path or "dofa" in name_or_path or "ssl4eo" in name_or_path:
                    backbone = make_terratorch_prithvi_dofa_slimable(backbone)
                elif 'terramind' in name_or_path:
                    backbone = make_terratorch_terramind_slimable(backbone)
            
            extract_multiscale_features(model=backbone, dm=dm, model_name=name_or_path, dataset_name=dataset)
            print(f"{dataset} extracted for {name_or_path}")

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SegmentationHead(nn.Module):
    """
    Lightweight SegFormer-style MLP head for frozen backbone probing.
    
    Inputs:  (B, 4, C, H_p, W_p)  - pre-extracted multi-scale features
    Outputs: (B, num_classes, H, W) - full resolution segmentation logits
    """
    def __init__(self, in_channels: int, num_classes: int, embed_dim: int = 256, img_size: int = 224, patch_size: int = 16):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size

        # Project each of the 4 layers to common embed_dim
        self.projections = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_channels, embed_dim),
                nn.ReLU(),
            ) for _ in range(4)
        ])

        # Fuse 4 projected features
        self.fuse = nn.Sequential(
            nn.Linear(4 * embed_dim, embed_dim),
            nn.ReLU(),
        )

        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 4, C, H_p, W_p)
        B, _, C, H_p, W_p = x.shape

        # Project each layer independently
        projected = []
        for i, proj in enumerate(self.projections):
            xi = x[:, i]                              # (B, C, H_p, W_p)
            xi = xi.permute(0, 2, 3, 1)              # (B, H_p, W_p, C)
            xi = proj(xi)                             # (B, H_p, W_p, embed_dim)
            projected.append(xi)

        # Fuse across layers
        x = torch.cat(projected, dim=-1)             # (B, H_p, W_p, 4*embed_dim)
        x = self.fuse(x)                             # (B, H_p, W_p, embed_dim)
        x = self.classifier(x)                       # (B, H_p, W_p, num_classes)

        # Upsample to full resolution
        x = x.permute(0, 3, 1, 2)                   # (B, num_classes, H_p, W_p)
        x = F.interpolate(x, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)

        return x                                      # (B, num_classes, H, W)

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from torchmetrics.segmentation import MeanIoU
from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score


def train(
    model: nn.Module,
    features: np.ndarray,
    labels: np.ndarray,
    epochs: int = 30,
    lr: float = 0.01,
    batch_size: int = 32,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> dict:
    """
    Train a segmentation head on pre-extracted features.

    Args:
        model:      SegmentationHead
        features:   (N, 4, C, H_p, W_p)
        labels:     (N, 1, H, W)
        epochs:     Number of training epochs
        lr:         Learning rate for AdamW
        batch_size: Batch size
        device:     Device to train on

    Returns:
        Dict with training history (loss per epoch)
    """
    features_t = torch.from_numpy(features).float()
    labels_t = torch.from_numpy(labels).long().squeeze(1)   # (N, H, W)

    dataset = TensorDataset(features_t, labels_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=4)

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(ignore_index=255)

    history = {"loss": []}
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)                           # (B, num_classes, H, W)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * x.size(0)

        scheduler.step()
        epoch_loss /= len(dataset)
        history["loss"].append(epoch_loss)
        print(f"  Training epoch {epoch+1:3d}/{epochs} | loss: {epoch_loss:.4f}")

    return history


def test(
    model: nn.Module,
    features: np.ndarray,
    labels: np.ndarray,
    batch_size: int = 32,
    num_classes: int = 7,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> dict:
    """
    Evaluate a segmentation head on pre-extracted features.

    Args:
        model:        SegmentationHead
        features:     (N, 4, C, H_p, W_p)
        labels:       (N, 1, H, W)
        batch_size:   Batch size
        num_classes:  Number of semantic classes
        ignore_index: Label value to ignore (e.g. 255 for unlabeled)
        device:       Device to run on

    Returns:
        Dict with mIoU, per-class IoU, accuracy, and F1
    """
    print("  Testing")
    features_t = torch.from_numpy(features).float()
    labels_t = torch.from_numpy(labels).long().squeeze(1)   # (N, H, W)

    dataset = TensorDataset(features_t, labels_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=4)

    model = model.to(device)
    model.eval()

    miou_metric = MeanIoU(num_classes=num_classes).to(device)
    acc_metric = MulticlassAccuracy(num_classes=num_classes, average="micro").to(device)
    f1_metric = MulticlassF1Score(num_classes=num_classes, average="macro").to(device)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)                           # (B, num_classes, H, W)
            preds = logits.argmax(dim=1)                # (B, H, W)
            miou_metric.update(preds, y)
            acc_metric.update(preds, y)
            f1_metric.update(preds, y)

    per_class_iou = miou_metric.compute()               # (num_classes,)
    metrics = {
        "mIoU":          per_class_iou.mean().item(),
        "per_class_iou": per_class_iou.cpu().tolist(),
        "accuracy":      acc_metric.compute().item(),
        "macro_f1":      f1_metric.compute().item(),
    }

    print(f"  mIoU: {metrics['mIoU']:.4f} | Acc: {metrics['accuracy']:.4f} | F1: {metrics['macro_f1']:.4f}")
    return metrics

In [7]:
import numpy as np
import pandas as pd


model_channel_data = [
    #('prithvi2_300', 1024, 16), 
    #('prithvi2_600', 1280, 14), 
    #('dofa', 768, 16), 
    #('dofa_large', 1024, 16), 
    #('terramind', 768, 16), 
    #('terramind_large', 1024, 16),
    #('ssl4eo_moco', 384, 16),
    #('ssl4eo_dino', 384, 16),
    ('panopticon', 768, 14), 
    ('copernicusFM', 768, 16), 
]


eval_segmentation = False
if eval_segmentation:
    for dataset in datasets:
        for name, d, p in model_channel_data:
            results = []
            csv_path = f"{BASE_PATH}/{name}/segmentation_scores_probe_{dataset}.csv"
            for scale in SCALES:
                print(f"{name:>15} : {scale:5.3f}")
                print('-'*(15+3+5))
                head = SegmentationHead(in_channels=d, num_classes=7 if dataset == 'm-cashew-plant' else 10, embed_dim=256, img_size=224, patch_size=p)
        
                print("Loading training data")
                train_features = np.load(f"{BASE_PATH}/{name}/scale_{scale}/features_{dataset}_train.npy")
                train_labels = np.load(f"{BASE_PATH}/{name}/scale_{scale}/labels_{dataset}_train.npy")
                train(model=head, features=train_features, labels=train_labels, epochs=30, lr=0.01)
                
                print("Loading test data")
                test_features = np.load(f"{BASE_PATH}/{name}/scale_{scale}/features_{dataset}_test.npy")
                test_labels = np.load(f"{BASE_PATH}/{name}/scale_{scale}/labels_{dataset}_test.npy")
                metrics = test(model=head, features=test_features, labels=test_labels, num_classes=7 if dataset == 'm-cashew-plant' else 10)
            
                print("Saving results")
                results.append({
                    "model": name,
                    "scale": scale,
                    "dataset": dataset,
                    "mIoU": metrics["mIoU"],
                    "accuracy": metrics["accuracy"],
                    "macro_f1": metrics["macro_f1"],
                })
        
                # Write header only if file doesn't exist yet
                df_row = pd.DataFrame(results[-1:])
                df_row.to_csv(csv_path, mode='a', index=False, header=not Path(csv_path).exists())
                print()
        print(f"Segmentation complete for {dataset} and {name}")

## Change Detection

### Preprocess

In [8]:
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from pathlib import Path

TEST_CITIES = ['brasilia','montpellier','norcia','rio','saclay_w','valencia','dubai','lasvegas','milano','chongqing']
TRAIN_CITIES = ['aguasclaras','bercy','bordeaux','nantes','paris','rennes','saclay_e','abudhabi','cupertino','pisa','beihai','hongkong','beirut','mumbai']
OSCD_BANDS = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B10', 'B11', 'B12']


def preprocess_oscd(
    root: str | Path,
    out_root: str | Path,
    crop_size: int = 256,
    train_stride: int = 128,
):
    """
    Crop all OSCD images and labels into fixed-size tiles and save as GeoTIFF.

    Output structure:
        out_root/
            train/
                abudhabi_0_0/
                    img1.tif   (C, crop_size, crop_size)
                    img2.tif
                    label.tif  (crop_size, crop_size)
                abudhabi_0_128/
                    ...
            test/
                brasilia_0_0/
                    ...
    """
    root = Path(root)
    out_root = Path(out_root)
    out_root.mkdir(exist_ok=True)

    img_root   = root / 'Onera Satellite Change Detection dataset - Images'
    train_root = root / 'Onera Satellite Change Detection dataset - Train Labels'
    test_root  = root / 'Onera Satellite Change Detection dataset - Test Labels'

    splits = [
        ('train', TRAIN_CITIES, train_root, train_stride),
        ('test',  TEST_CITIES,  test_root,  crop_size),    # no overlap for test
    ]

    for split, cities, label_root, stride in splits:
        for city in cities:
            print(f"Processing {split}/{city}...")

            # Read all 13 bands for both times (save all bands, select at load time)
            img1 = read_image(img_root / city, '1', OSCD_BANDS)  # (13, H, W)
            img2 = read_image(img_root / city, '2', OSCD_BANDS)  # (13, H, W)
            label = read_label(label_root, city)                  # (H, W)

            H, W = label.shape
            crops = get_crops(H, W, crop_size, stride)

            for r, c in crops:
                crop_dir = out_root / split / f'{city}_{r}_{c}'
                crop_dir.mkdir(parents=True, exist_ok=True)

                _save_tif(img1[:, r:r+crop_size, c:c+crop_size], crop_dir / 'img1.tif')
                _save_tif(img2[:, r:r+crop_size, c:c+crop_size], crop_dir / 'img2.tif')
                _save_tif(label[None, r:r+crop_size, c:c+crop_size], crop_dir / 'label.tif')

def read_image(city_dir: Path, time: str, bands: list[str]) -> np.ndarray:
    """Read selected bands from imgs_{time}_rect, return (C, H, W) float32."""
    rect_dir = city_dir / f'imgs_{time}_rect'
    arrays = []
    for band in bands:
        tif = rect_dir / f'{band}.tif'
        with rasterio.open(tif) as src:
            arrays.append(src.read(1).astype(np.float32))
    return np.stack(arrays, axis=0)   # (C, H, W)

def read_label(label_dir: Path, city: str) -> np.ndarray:
    """Read binary change map, return (H, W) uint8 with values 0/1."""
    tif = label_dir / city / 'cm' / f'{city}-cm.tif'
    with rasterio.open(tif) as src:
        cm = src.read(1)
    return (cm != 1).astype(np.uint8)  # binarize 1/2 → 0/1

def get_crops(h: int, w: int, crop_size: int, stride: int) -> list[tuple]:
    """Generate (row, col) top-left corners for all valid crops."""
    crops = []
    for r in range(0, h - crop_size + 1, stride):
        for c in range(0, w - crop_size + 1, stride):
            crops.append((r, c))
    # Ensure we always cover the bottom-right corner
    crops.append((h - crop_size, w - crop_size))
    return list(set(crops))

def _save_tif(arr: np.ndarray, path: Path):
    """Save (C, H, W) or (1, H, W) array as GeoTIFF."""
    c, h, w = arr.shape
    with rasterio.open(
        path, 'w',
        driver='GTiff',
        height=h, width=w,
        count=c,
        dtype=arr.dtype,
    ) as dst:
        dst.write(arr)

# preprocess_oscd(root="./data/OSCD", out_root="./data/OSCD_preprocessed", crop_size=224, train_stride=112)

### Features

In [9]:
def compute_normalization_stats(
    root: str | Path,
    bands: list[str],
) -> tuple[np.ndarray, np.ndarray]:
    band_indices = [OSCD_BANDS.index(b) + 1 for b in bands]
    train_dir = Path(root) / 'train'
    per_file_means, per_file_vars, per_file_counts = [], [], []

    for sample_dir in sorted(train_dir.iterdir()):
        for fname in ('img1.tif', 'img2.tif'):
            with rasterio.open(sample_dir / fname) as src:
                img = src.read(band_indices).astype(np.float64)
            n = img.shape[1] * img.shape[2]
            per_file_means.append(img.mean(axis=(1, 2)))
            per_file_vars.append(img.var(axis=(1, 2)))
            per_file_counts.append(n)

    counts = np.array(per_file_counts)
    means  = np.array(per_file_means)
    vars_  = np.array(per_file_vars)
    total_n    = counts.sum()
    total_mean = (counts[:, None] * means).sum(axis=0) / total_n
    total_var  = (counts[:, None] * (vars_ + (means - total_mean) ** 2)).sum(axis=0) / total_n
    mean = total_mean.astype(np.float32)
    std  = np.sqrt(total_var).astype(np.float32)

    # Print ready-to-paste constant
    print("OSCD_NORM_STATS = {")
    for b, m, s in zip(bands, mean, std):
        print(f"    '{b}': ({m:.4f}, {s:.4f}),")
    print("}")

    return mean, std

# compute_normalization_stats(root, OSCD_BANDS)

In [10]:
from torch.utils.data import Dataset

OSCD_NORM_STATS = {
    'B01': (1592.9020, 281.0677),
    'B02': (1397.6360, 431.9554),
    'B03': (1328.5879, 558.9767),
    'B04': (1365.8481, 791.0626),
    'B05': (1491.9292, 743.7421),
    'B06': (1899.7970, 759.8222),
    'B07': (2115.0527, 838.9940),
    'B08': (2025.9111, 860.4077),
    'B8A': (2244.3398, 905.1946),
    'B09': (656.9305, 320.6088),
    'B10': (14.2359, 8.1941),
    'B11': (2023.4193, 1019.0800),
    'B12': (1487.6002, 875.3322),
}

class OSCDDataset(Dataset):
    def __init__(
        self,
        root: str | Path,
        split: str,
        bands: list[str],
        mean_std: tuple[np.ndarray, np.ndarray] | None = None,
        input_size: int = 224,
    ):
        self.root = Path(root)
        self.bands = [OSCD_BANDS.index(b) + 1 for b in bands]
        self.input_size = input_size
        self.samples = sorted((self.root / split).iterdir())

        if mean_std is not None:
            mean, std = mean_std
            self.mean = mean[:, None, None].astype(np.float32)  # (C, 1, 1) for broadcasting
            self.std  = std[:, None, None].astype(np.float32)
        else:
            self.mean = self.std = None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample_dir = self.samples[idx]

        with rasterio.open(sample_dir / 'img1.tif') as src:
            img1 = src.read(self.bands).astype(np.float32)
        with rasterio.open(sample_dir / 'img2.tif') as src:
            img2 = src.read(self.bands).astype(np.float32)
        with rasterio.open(sample_dir / 'label.tif') as src:
            label = src.read(1)

        if self.mean is not None:
            img1 = (img1 - self.mean) / self.std
            img2 = (img2 - self.mean) / self.std

        img1  = self._resize(img1, 'bilinear')
        img2  = self._resize(img2, 'bilinear')
        label = self._resize(label[None].astype(np.float32), 'nearest')[0]

        return {
            'image1': torch.from_numpy(img1),
            'image2': torch.from_numpy(img2),
            'label':  torch.from_numpy(label).long(),
        }

    def _resize(self, arr: np.ndarray, mode: str) -> np.ndarray:
        if arr.shape[-1] == self.input_size:
            return arr
        t = torch.from_numpy(arr).unsqueeze(0)
        t = F.interpolate(t, size=(self.input_size, self.input_size), mode=mode,
                          align_corners=False if mode == 'bilinear' else None)
        return t.squeeze(0).numpy()
        
def get_mean_std(bands: list[str]) -> tuple[np.ndarray, np.ndarray]:
    mean = np.array([OSCD_NORM_STATS[b][0] for b in bands], dtype=np.float32)
    std  = np.array([OSCD_NORM_STATS[b][1] for b in bands], dtype=np.float32)
    return mean, std

root="./data/OSCD_preprocessed"

In [11]:
# All bands available in OSCD (imgs_X_rect)
OSCD_AVAILABLE_BANDS = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 
                         'B07', 'B08', 'B8A', 'B09', 'B10', 'B11', 'B12']

def get_oscd_bands_for_model(model_name: str) -> list[str]:
    """
    Select bands for OSCD by intersecting the model's translated_bands
    with what OSCD has available, preserving model band order.
    """
    translated = evaluated_models[model_name]["translated_bands"]
    return [b for b in translated if b in OSCD_AVAILABLE_BANDS]


# Check what each model gets
for name in ['prithvi2_300', 'prithvi2_600', 'dofa', 'dofa_large', 'terramind', 'terramind_large', 'ssl4eo_moco', 'ssl4eo_dino', 'panopticon', 'copernicusFM']:
    bands = get_oscd_bands_for_model(name)
    print(f"{name:>15}: {bands}")

   prithvi2_300: ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12']
   prithvi2_600: ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12']
           dofa: ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']
     dofa_large: ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']
      terramind: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12']
terramind_large: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12']
    ssl4eo_moco: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B10', 'B11', 'B12']
    ssl4eo_dino: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B10', 'B11', 'B12']
     panopticon: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12']
   copernicusFM: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12']


In [12]:
import warnings
import rasterio
from rasterio.errors import NotGeoreferencedWarning

warnings.filterwarnings('ignore', category=NotGeoreferencedWarning)

def get_multiscale_features_and_labels_oscd(model, dl, ps, device):
    """
    Extract features from an OSCD dataloader returning {'image1', 'image2', 'label'}.
    
    Returns:
        features: (N, 2, 4, C, H_p, W_p)  - stacked t1/t2 features
        labels:   (N, H, W)
    """
    model.to(device)
    model.eval()

    all_features, all_labels = [], []

    for batch in tqdm(dl, desc="Extracting features", leave=False):
        img1 = batch['image1'].to(device)
        img2 = batch['image2'].to(device)
        y    = batch['label']

        with torch.no_grad():
            feat1 = select_four_layers(model(img1), img_size=224, patch_size=ps)
            feat2 = select_four_layers(model(img2), img_size=224, patch_size=ps)

        feat1 = F.normalize(feat1, dim=-1).cpu()
        feat2 = F.normalize(feat2, dim=-1).cpu()

        all_features.append(torch.stack([feat1, feat2], dim=1))  # (B, 2, 4, C, H_p, W_p)
        all_labels.append(y)

    return (
        torch.cat(all_features, dim=0).numpy(),   # (N, 2, 4, C, H_p, W_p)
        torch.cat(all_labels,   dim=0).numpy(),   # (N, H, W)
    )


def extract_multiscale_features_oscd(model, model_name, ps, base_path, root, device):
    """
    Extract and save multi-scale features for all OSCD splits and scales.
    """
    bands = get_oscd_bands_for_model(model_name)
    mean_std = get_mean_std(bands)

    for scale in tqdm(SCALES, desc="Iterating scales"):
        print(f"{model_name} : {scale}")

        try:
            model.backbone.set_to_scale(scale)
        except AttributeError:
            try:
                model.set_to_scale(scale)
            except AttributeError:
                model.model.set_to_scale(scale)

        for split in ['train', 'test']:
            ds = OSCDDataset(root=root, split=split, bands=bands, mean_std=mean_std, input_size=224)
            dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=16, pin_memory=True)

            features, labels = get_multiscale_features_and_labels_oscd(model, dl, ps, device)

            out_dir = Path(base_path) / model_name / f"scale_{scale}"
            out_dir.mkdir(parents=True, exist_ok=True)
            np.save(out_dir / f"features_oscd_{split}.npy", features)
            np.save(out_dir / f"labels_oscd_{split}.npy",   labels)

extract_cd_features = False
OSCD_ROOT = "/scratch/htc/lhackel/data/OSCD_preprocessed"

if extract_cd_features:
    for name_or_path in ['copernicusFM']:
        ps = 14 if name_or_path in ['prithvi2_600', 'panopticon'] else 16

        if name_or_path == 'panopticon':
            import panopticon_helpers
            from torchgeo.models import panopticon_vitb14, Panopticon_Weights
            
            pano = panopticon_vitb14(weights=Panopticon_Weights.VIT_BASE14, img_size=224)
            backbone = panopticon_helpers.PanopticonBackbone(pano.model, chn_ids,
                                                             feature_mode="blocks")
            backbone = panopticon_helpers.make_panopticon_slimable(backbone)
        elif name_or_path == 'copernicusFM':
            import copernicusFM_helpers
            from copernicusFM import vit_base_patch16
            COPERNICUSFM_CHECKPOINTS = [
                ("/home/leonard/data/model-weights/CopernicusFM_ViT_base_varlang_e100.pth", "copernicusFM"),
                ("/scratch/htc/lhackel/model-weights/CopernicusFM_ViT_base_varlang_e100.pth", "copernicusFM"),
            ]

            checkpoint_path, version = next(
                ((Path(c), n) for c, n in COPERNICUSFM_CHECKPOINTS if Path(c).exists()),
                (None, None)
            )

            if checkpoint_path is None:
                raise FileNotFoundError(f"No CopernicusFM checkpoint found in: {COPERNICUSFM_CHECKPOINTS}")
            checkpoint_path = checkpoint_path.resolve(strict=True)

            keep_idx, wvs, bws = [], [], []
            for i, b in enumerate(bandnames):
                params = copernicusFM_helpers.copernicusFM_band_params(b, include_sar=False)
                if params is None:
                    continue
                keep_idx.append(i)
                wv, bw = params
                wvs.append(wv)
                bws.append(bw)
            assert keep_idx, f"no usable CopernicusFM channels for {dataset}"

            backbone = vit_base_patch16(num_classes=10, global_pool=False)
            checkpoint = torch.load(checkpoint_path, map_location="cpu")
            state_dict = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint
            msg = backbone.load_state_dict(state_dict, strict=False)
            #print(msg)

            backbone = make_terratorch_prithvi_dofa_slimable(backbone)  # CopernicusFM is similar enough to DOFA
            backbone = copernicusFM_helpers.CopernicusFMBackbone(backbone, wvs, bws, kernel_size=16, input_mode="spectral")
            backbone.get_all_features(True)
        else:
            backbone = BACKBONE_REGISTRY.build(**evaluated_models[name_or_path]["kwargs"])
            if "prithvi" in name_or_path or "dofa" in name_or_path or "ssl4eo" in name_or_path:
                backbone = make_terratorch_prithvi_dofa_slimable(backbone)
            elif 'terramind' in name_or_path:
                backbone = make_terratorch_terramind_slimable(backbone)
    
        extract_multiscale_features_oscd(
            model=backbone,
            model_name=name_or_path,
            ps=ps,
            base_path=BASE_PATH,
            root=OSCD_ROOT,
            device=device,
        )

### Eval

In [13]:
def train_binary(
    model: nn.Module,
    features: np.ndarray,
    labels: np.ndarray,
    epochs: int = 30,
    lr: float = 0.01,
    batch_size: int = 32,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
    weight_decay: float = 0.01,
    seed: int = 42
) -> dict:
    pl.seed_everything(seed)
    features_t = torch.from_numpy(features).float()
    labels_t = torch.from_numpy(labels).long().squeeze(1)   # (N, H, W)

    # Compute class weights from label distribution
    n_neg = (labels_t == 0).sum().item()
    n_pos = (labels_t == 1).sum().item()
    if n_pos > 0 and n_neg > 0:
        class_weights = torch.tensor([1.0, n_neg / n_pos], device=device)
    else:
        class_weights = None

    dataset = TensorDataset(features_t, labels_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=8, persistent_workers=True)

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(ignore_index=255, weight=class_weights)

    history = {"loss": []}
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * x.size(0)

        scheduler.step()
        epoch_loss /= len(dataset)
        history["loss"].append(epoch_loss)
        print(f"  Training epoch {epoch+1:3d}/{epochs} | loss: {epoch_loss:.4f}")

    return history

from torchmetrics.classification import BinaryJaccardIndex, BinaryF1Score, BinaryAccuracy

def test_binary(
    model: nn.Module,
    features: np.ndarray,
    labels: np.ndarray,
    batch_size: int = 32,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> dict:
    features_t = torch.from_numpy(features).float()
    labels_t   = torch.from_numpy(labels).long().squeeze(1)   # (N, H, W)

    dataset = TensorDataset(features_t, labels_t)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=4)

    model = model.to(device)
    model.eval()

    iou_change  = BinaryJaccardIndex().to(device)   # IoU of change class (class 1)
    iou_nochange = BinaryJaccardIndex().to(device)  # IoU of no-change class (class 0)
    f1_metric   = BinaryF1Score().to(device)
    acc_metric  = BinaryAccuracy().to(device)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)           # (B, H, W)
            iou_change.update(preds, y)
            iou_nochange.update(1 - preds, 1 - y)   # flip for no-change class
            f1_metric.update(preds, y)
            acc_metric.update(preds, y)

    change_iou   = iou_change.compute().item()
    nochange_iou = iou_nochange.compute().item()
    metrics = {
        "mIoU":       (change_iou + nochange_iou) / 2,
        "change_iou":   change_iou,
        "nochange_iou": nochange_iou,
        "accuracy":   acc_metric.compute().item(),
        "macro_f1":   f1_metric.compute().item(),
    }
    print(f"  mIoU: {metrics['mIoU']:.4f} | Change IoU: {change_iou:.4f} | F1: {metrics['macro_f1']:.4f} | Acc: {metrics['accuracy']:.4f}")
    return metrics

import numpy as np
import pandas as pd

results = []

model_channel_data = [
    #('prithvi2_300', 1024, 16), 
    #('prithvi2_600', 1280, 14), 
    #('dofa', 768, 16), 
    #('dofa_large', 1024, 16), 
    #('terramind', 768, 16), 
    #('terramind_large', 1024, 16),
    #('ssl4eo_moco', 384, 16),
    #('ssl4eo_dino', 384, 16),
    ('panopticon', 768, 14), 
    ('copernicusFM', 768, 16), 
]

eval_change_detection = True
if eval_change_detection:
    for name, d, p in model_channel_data:
        csv_path = f"{BASE_PATH}/{name}/change-detection_scores_probe_oscd.csv"
        for scale in SCALES:
            print(f"{name:>15} : {scale:5.3f}")
            print('-' * (15 + 3 + 5))

            head = SegmentationHead(in_channels=d, num_classes=2, embed_dim=128, img_size=224, patch_size=p)

            print("Loading training data")
            train_features = np.load(f"{BASE_PATH}/{name}/scale_{scale}/features_oscd_train.npy")  # (N, 2, 4, C, H_p, W_p)
            train_labels   = np.load(f"{BASE_PATH}/{name}/scale_{scale}/labels_oscd_train.npy")    # (N, H, W)
            train_diff = train_features[:, 1] - train_features[:, 0]                               # (N, 4, C, H_p, W_p)

            train_binary(model=head, features=train_diff, labels=train_labels[:, None], epochs=30, lr=0.01, weight_decay=0.1)  # train_labels[:, None] to add a label dim

            print("Loading test data")
            test_features = np.load(f"{BASE_PATH}/{name}/scale_{scale}/features_oscd_test.npy")
            test_labels   = np.load(f"{BASE_PATH}/{name}/scale_{scale}/labels_oscd_test.npy")
            test_diff = test_features[:, 1] - test_features[:, 0]

            metrics = test_binary(model=head, features=test_diff, labels=test_labels[:, None])
            
            results.append({
                "model": name,
                "scale": scale,
                "dataset": "oscd",
                **metrics
            })

            df_row = pd.DataFrame(results[-1:])
            df_row.to_csv(csv_path, mode='a', index=False, header=not Path(csv_path).exists())
            print()

    print("Change detection complete")

     panopticon : 0.001
-----------------------
Loading training data


[rank: 0] Seed set to 42


  Training epoch   1/30 | loss: 0.6810
  Training epoch   2/30 | loss: 0.5435
  Training epoch   3/30 | loss: 0.4462
  Training epoch   4/30 | loss: 0.3496
  Training epoch   5/30 | loss: 0.2844
  Training epoch   6/30 | loss: 0.2395
  Training epoch   7/30 | loss: 0.2064
  Training epoch   8/30 | loss: 0.1787
  Training epoch   9/30 | loss: 0.1656
  Training epoch  10/30 | loss: 0.1490
  Training epoch  11/30 | loss: 0.1352
  Training epoch  12/30 | loss: 0.1240
  Training epoch  13/30 | loss: 0.1140
  Training epoch  14/30 | loss: 0.1078
  Training epoch  15/30 | loss: 0.1081
  Training epoch  16/30 | loss: 0.1013
  Training epoch  17/30 | loss: 0.0961
  Training epoch  18/30 | loss: 0.0927
  Training epoch  19/30 | loss: 0.0903
  Training epoch  20/30 | loss: 0.0869
  Training epoch  21/30 | loss: 0.0843
  Training epoch  22/30 | loss: 0.0830
  Training epoch  23/30 | loss: 0.0809
  Training epoch  24/30 | loss: 0.0794
  Training epoch  25/30 | loss: 0.0791
  Training epoch  26/30 |

[rank: 0] Seed set to 42


  Training epoch   1/30 | loss: 0.6820
  Training epoch   2/30 | loss: 0.4638
  Training epoch   3/30 | loss: 0.3339
  Training epoch   4/30 | loss: 0.2613
  Training epoch   5/30 | loss: 0.2248
  Training epoch   6/30 | loss: 0.1991
  Training epoch   7/30 | loss: 0.1827
  Training epoch   8/30 | loss: 0.1634
  Training epoch   9/30 | loss: 0.1474
  Training epoch  10/30 | loss: 0.1353
  Training epoch  11/30 | loss: 0.1244
  Training epoch  12/30 | loss: 0.1159
  Training epoch  13/30 | loss: 0.1097
  Training epoch  14/30 | loss: 0.1046
  Training epoch  15/30 | loss: 0.0988
  Training epoch  16/30 | loss: 0.0934
  Training epoch  17/30 | loss: 0.0872
  Training epoch  18/30 | loss: 0.0839
  Training epoch  19/30 | loss: 0.0816
  Training epoch  20/30 | loss: 0.0793
  Training epoch  21/30 | loss: 0.0775
  Training epoch  22/30 | loss: 0.0762
  Training epoch  23/30 | loss: 0.0747
  Training epoch  24/30 | loss: 0.0736
  Training epoch  25/30 | loss: 0.0733
  Training epoch  26/30 |

...

  Training epoch   1/30 | loss: 0.6693
  Training epoch   2/30 | loss: 0.3997
  Training epoch   3/30 | loss: 0.2796
  Training epoch   4/30 | loss: 0.2226
  Training epoch   5/30 | loss: 0.1786
  Training epoch   6/30 | loss: 0.1505
  Training epoch   7/30 | loss: 0.1340
  Training epoch   8/30 | loss: 0.1209
  Training epoch   9/30 | loss: 0.1124
  Training epoch  10/30 | loss: 0.1065
  Training epoch  11/30 | loss: 0.1002
  Training epoch  12/30 | loss: 0.0965
  Training epoch  13/30 | loss: 0.0910
  Training epoch  14/30 | loss: 0.0881
  Training epoch  15/30 | loss: 0.0864
  Training epoch  16/30 | loss: 0.0841
  Training epoch  17/30 | loss: 0.0817
  Training epoch  18/30 | loss: 0.0793
  Training epoch  19/30 | loss: 0.0782
  Training epoch  20/30 | loss: 0.0765
  Training epoch  21/30 | loss: 0.0755
  Training epoch  22/30 | loss: 0.0750
  Training epoch  23/30 | loss: 0.0740
  Training epoch  24/30 | loss: 0.0736
  Training epoch  25/30 | loss: 0.0737
  Training epoch  26/30 |

[rank: 0] Seed set to 42


  Training epoch   1/30 | loss: 0.6758
  Training epoch   2/30 | loss: 0.4136
  Training epoch   3/30 | loss: 0.2834
  Training epoch   4/30 | loss: 0.2196
  Training epoch   5/30 | loss: 0.1774
  Training epoch   6/30 | loss: 0.1479
  Training epoch   7/30 | loss: 0.1322
  Training epoch   8/30 | loss: 0.1234
  Training epoch   9/30 | loss: 0.1127
  Training epoch  10/30 | loss: 0.1057
  Training epoch  11/30 | loss: 0.0989
  Training epoch  12/30 | loss: 0.0934
  Training epoch  13/30 | loss: 0.0900
  Training epoch  14/30 | loss: 0.0903
  Training epoch  15/30 | loss: 0.0892
  Training epoch  16/30 | loss: 0.0895
  Training epoch  17/30 | loss: 0.0865
  Training epoch  18/30 | loss: 0.0844
  Training epoch  19/30 | loss: 0.0816
  Training epoch  20/30 | loss: 0.0801
  Training epoch  21/30 | loss: 0.0786
  Training epoch  22/30 | loss: 0.0779
  Training epoch  23/30 | loss: 0.0767
  Training epoch  24/30 | loss: 0.0758
  Training epoch  25/30 | loss: 0.0758
  Training epoch  26/30 |

[rank: 0] Seed set to 42


  Training epoch   1/30 | loss: 0.6698
  Training epoch   2/30 | loss: 0.4006
  Training epoch   3/30 | loss: 0.2809
  Training epoch   4/30 | loss: 0.2106
  Training epoch   5/30 | loss: 0.1802
  Training epoch   6/30 | loss: 0.1521
  Training epoch   7/30 | loss: 0.1299
  Training epoch   8/30 | loss: 0.1157
  Training epoch   9/30 | loss: 0.1051
  Training epoch  10/30 | loss: 0.0996
  Training epoch  11/30 | loss: 0.0945
  Training epoch  12/30 | loss: 0.0909
  Training epoch  13/30 | loss: 0.0907
  Training epoch  14/30 | loss: 0.0867
  Training epoch  15/30 | loss: 0.0856
  Training epoch  16/30 | loss: 0.0835
  Training epoch  17/30 | loss: 0.0804
  Training epoch  18/30 | loss: 0.0782
  Training epoch  19/30 | loss: 0.0764
  Training epoch  20/30 | loss: 0.0749
  Training epoch  21/30 | loss: 0.0739
  Training epoch  22/30 | loss: 0.0734
  Training epoch  23/30 | loss: 0.0724
  Training epoch  24/30 | loss: 0.0717
  Training epoch  25/30 | loss: 0.0718
  Training epoch  26/30 |

In [22]:
metric = 'macro_f1'
df = pd.read_csv(csv_path)
df = df[df.scale.isin(SCALES)]
df['max_val'] = df.groupby('model')[metric].transform('max')
df['rel_ret_rate'] = df[metric]/df['max_val']

df['rel_ret_rate_p'] = df['rel_ret_rate']*100
df['rel_ret_rate_p'] = df['rel_ret_rate_p'].apply(lambda x: f"{x:4.1f}%")
pivot = df.pivot(index='model', columns='scale', values='rel_ret_rate_p')
print(pivot.to_markdown())

| model           | 0.001   | 0.01   | 0.1    | 0.25   | 0.5   | 1.0    |
|:----------------|:--------|:-------|:-------|:-------|:------|:-------|
| dofa            | 97.3%   | 95.4%  | 100.0% | 94.9%  | 98.8% | 64.0%  |
| dofa_large      | 100.0%  | 98.6%  | 89.7%  | 99.6%  | 94.5% | 57.5%  |
| prithvi2_300    | 87.5%   | 89.5%  | 92.5%  | 100.0% | 96.3% | 78.3%  |
| prithvi2_600    | 88.9%   | 95.9%  | 100.0% | 87.4%  | 91.9% | 94.5%  |
| terramind       | 85.7%   | 85.5%  | 94.0%  | 70.7%  | 98.9% | 100.0% |
| terramind_large | 90.3%   | 80.1%  | 99.1%  | 99.0%  | 95.1% | 100.0% |
